# Build new PDFF phenotype file

In [93]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [107]:
xmdu_old_data = pd.read_table('/mnt/speliotes-lab/Projects/UK_ATLAS/IndivProj/Xiaomeng/SAIGE/origdata/PDFF_4_7/pheno_pc_age_sex.allwhite.PDFF_4_7.txt')
print(xmdu_old_data.shape)

(43293, 18)


In [ ]:
new_pdff = pd.read_table('/mnt/speliotes-lab/Projects/UK_ATLAS/Liver_MRI/PDFF/data/UKBB_liver_MRI_New_GRE_and_New_IDEAL_PDFF_combine.fillmissing.tab.2023.txt')
print(new_pdff.shape)
new_pdff

In [ ]:
eid2new_pdff = {int(e):p for e,p in zip(new_pdff.eid, new_pdff.true_PDFF_40061) if not np.isnan(p)}
print(len(eid2new_pdff))
xmdu_old_data["PDFF_new_real"] = [eid2new_pdff[int(e)] if int(e) in eid2new_pdff else np.nan for e in xmdu_old_data.eid]
xmdu_old_data

In [110]:
sum([e not in eid2new_pdff for e in xmdu_old_data.eid])

3815

Seems like 10 people from the earlier PDFF analysis dropped out

In [111]:
xmdu_new_pdff_real_data = xmdu_old_data[[not np.isnan(p) for p in xmdu_old_data.PDFF_new_real]]
print(xmdu_new_pdff_real_data.shape)

(39478, 19)


In [20]:
#xmdu_new_pdff_real_data.to_csv('/home/craut/wkspce/new_MRI_PDFF/data_out/xmdu_new_pdff_real_results/new_pdff_real_data.txt', index=False,sep='\t')

# Evaluate the efficacy of the old predicted PDFF values against ground truth

In [21]:
# calculate RMSE for PDFF_new_real vs PDFF
print(np.sqrt(np.mean((xmdu_new_pdff_real_data.PDFF_new_real - xmdu_new_pdff_real_data.PDFF)**2)))

2.090493159306228


In [ ]:
# grab the list of individuals imaged through the gre protocol
gre_list = pd.read_table("/mnt/speliotes-lab/Projects/UK_ATLAS/Liver_MRI/GRE/segmentation/data/GRE_id.list",header=None)
gre_list

In [24]:
gre_eid_set = set(gre_list[0])
print(len(gre_eid_set))

10105


In [ ]:
gre_pat = xmdu_new_pdff_real_data[[e in gre_eid_set for e in xmdu_new_pdff_real_data.eid]]
gre_pat

In [26]:
# rmse for gre protocol only
print(np.sqrt(np.mean((gre_pat.PDFF_new_real - gre_pat.PDFF)**2)))

1.2265421230253637


In [ ]:
# filter anyone who was not gre --> ideal protocol
ideal_pat = xmdu_new_pdff_real_data[[e not in gre_eid_set for e in xmdu_new_pdff_real_data.eid]]
ideal_pat

In [28]:
# rmse for ideal protocol
print(np.sqrt(np.mean((ideal_pat.PDFF_new_real - ideal_pat.PDFF)**2)))

2.270272049486994


In [71]:
def make_scatter(true,pred,title="",color=None,eid=None,symbol=None):
    true = np.array(true)
    pred = np.array(pred)
    rmse = np.sqrt(np.mean((true-pred)**2))
    pearson_correlation = np.corrcoef(true,pred)[0,1]
    title = f"{title} (N={len(true)}, RMSE={rmse:.2f}, Pearson={pearson_correlation:.2f})"
    if not color is None:
        color = np.array([str(c) for c in color])
    fig = px.scatter(x=true,y=pred,color=color,title=title,opacity=0.5,hover_name=[f"eid:{e}" for e in eid],symbol=symbol)
    ylim = min(max(pred),max(true))
    fig.add_trace(px.line(x=[0, ylim], y=[0, ylim]).data[0])
    fig.update_traces(line=dict(dash="dash",color="black"))
    fig.update_layout(xaxis_title="True PDFF (%)", yaxis_title="ResNet Precited PDFF (%)")
    return fig

In [79]:
df = gre_pat
fig = make_scatter(df.PDFF,
                df.PDFF_new_real,
                "GRE true vs predicted PDFF scatterplot",
                ["white" if not np.isnan(w) else "not white" for w in df.white],
                df.eid,
                ["Male" if s == 1 else "Female" for s in df.sex])
fig.write_html("gre_true_vs_pred.html")

In [80]:
df = ideal_pat
fig = make_scatter(df.PDFF,
             df.PDFF_new_real,
             "IDEAL true vs predicted PDFF scatterplot",
             ["white" if not np.isnan(w) else "not white" for w in df.white],
             df.eid,
             ["Male" if s == 1 else "Female" for s in df.sex])
fig.write_html("IDEAL_true_vs_pred.html")

In [81]:
df = xmdu_new_pdff_real_data
fig = make_scatter(df.PDFF,
             df.PDFF_new_real,
             "GRE & IDEAL true vs predicted PDFF scatterplot",
             ["GRE" if int(e) in gre_eid_set else "IDEAL" for e in df.eid],
             df.eid,
             ["Male" if s == 1 else "Female" for s in df.sex])
fig.write_html("both_true_vs_pred.html")

# Test for Sex and White bias

We know that the UKBB is a predominantly white patient population. There should also be a pretty good balance of across the sex's. I just want to make sure that there is no major bias in the predictions made earlier which we can evaluate now due to the discovery of the true PDFF values.

In [119]:
# correct a few columns
updated_df = pd.DataFrame(xmdu_new_pdff_real_data)
updated_df["protocol"] = ["GRE" if int(e) in gre_eid_set else "IDEAL" for e in updated_df.eid]
updated_df["white_labeled"] = ["white" if not np.isnan(w) else "not_white" for w in xmdu_new_pdff_real_data.white]
updated_df["white_labeled"] = pd.Categorical(updated_df["white_labeled"], categories=["white","not_white"])
updated_df["sex_labeled"] = ["Male" if s == 1 else "Female" for s in xmdu_new_pdff_real_data.sex]
updated_df.sort_values(by=["protocol","sex_labeled","white_labeled"], ascending=[True,True,False],inplace=True)
updated_df.reset_index(drop=True,inplace=True)


# build 2-way table to establish counts of whites and sex 
df = updated_df
table = pd.crosstab([df["protocol"], df["white_labeled"]],df["sex_labeled"], margins=True, margins_name="Total")
table

sex_labeled             Female   Male  Total
protocol white_labeled                      
GRE      white            3882   3561   7443
         not_white         529    508   1037
IDEAL    white           13904  12986  26890
         not_white        2233   1875   4108
Total                    20548  18930  39478

So after talking with Antonio I think we want to test the following hypothesis:
- Is there an effect on error with the following?
  - sex
  - white
  - protocol
  - age

We can test this through the below model:
$$\text{true}-\text{predicted} = \beta_0 + \beta_1 \text{sex} + \beta_2 \text{white} + \beta_3 \text{protocol} + \beta_4 \text{age} + \epsilon$$
$$H_0: \beta_0 = \beta_1 = \beta_2 = \beta_3 = \beta_4$$
$$H_1: \text{any } \beta \neq 0$$



In [ ]:
df

In [120]:
y = np.array(df.PDFF_new_real) - np.array(df.PDFF)
df["y"] = y
mod = smf.ols(formula='y ~ sex_labeled + white_labeled + protocol + age', data=df)

res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     100.6
Date:                Thu, 03 Aug 2023   Prob (F-statistic):           2.43e-85
Time:                        18:54:37   Log-Likelihood:                -78235.
No. Observations:               39478   AIC:                         1.565e+05
Df Residuals:                   39473   BIC:                         1.565e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

We can see that there is a statistically significant association with sex and age with bias but not ancestry.